# Drug release modeling using physics-informed neural networks

Code for Qureshi, Shukla and Srivastava, "Drug release modeling using Physics-Informed Neural Networks", *Journal of Drug Delivery Science and Technology* 125 (2026) 108654, [doi:10.1016/j.jddst.2026.108654](https://doi.org/10.1016/j.jddst.2026.108654).

Run the cells in order. They fit the classical models, train the PINNs, run the noise study with a 50-PINN ensemble and an MC-dropout BPINN, then the limited-data study, and save Figures 3 to 9 to `figures/`.

## Setup

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from pyDOE import lhs
from sklearn.metrics import mean_squared_error, mean_absolute_error

# python random picks the ensemble seeds, so it is seeded too
random.seed(1234)
np.random.seed(1234)
tf.random.set_seed(1234)
tf.config.experimental.enable_op_determinism()

plt.rcParams.update({
    'font.size': 13,
    'axes.labelsize': 13,
    'axes.titlesize': 15,
    'legend.fontsize': 10,
    'figure.dpi': 150,
})

os.makedirs('figures', exist_ok=True)
print(f'TensorFlow {tf.__version__}')

## Data

Cumulative Rhodamine B release (ppm) from planar, 1D wrinkled and 2D crumpled graphene oxide films, 14 time points over 48 h (2880 min), from Liu et al., ACS Nano 15 (2021) 20105. Time is normalized to [0, 1].

In [ ]:
data = pd.read_csv('Data/data.csv')
display(data)

time_raw = data['time min'].values
time_normalized = time_raw / np.max(time_raw)

flat_film_ppm = data['flat film ppm'].values
film_1d_ppm = data['1D film ppm'].values
film_2d_ppm = data['2D film ppm'].values

t_col = time_normalized[:, np.newaxis]
u_flat = flat_film_ppm[:, np.newaxis]
u_1d = film_1d_ppm[:, np.newaxis]
u_2d = film_2d_ppm[:, np.newaxis]

film_labels = ['Flat', '1D', '2D']
film_data = [u_flat, u_1d, u_2d]
film_data_1d = [flat_film_ppm, film_1d_ppm, film_2d_ppm]

print(f'\nDataset: {len(time_raw)} time points over {time_raw[-1]:.0f} minutes ({time_raw[-1]/60:.0f} hours)')

## Classical models

Fit by nonlinear least squares.

| Model | Equation | Parameters |
|-------|----------|------------|
| Higuchi | $Q = A\sqrt{t} + B$ | $A$, $B$ |
| Peppas | $M_t/M_\infty = k \cdot t^n$ | $k$, $n$ |
| Fick | $C = C_0(1 - e^{-Dt})$ | $D$, $C_0$ |

In [ ]:
def higuchi_model(t, A, B):
    return A * np.sqrt(t) + B

def peppas_model(t, k, n):
    return k * (t ** n)

def fick_model(t, D, C0):
    return C0 * (1 - np.exp(-D * t))

def fit_models(time, concentration):
    popt_h, _ = curve_fit(higuchi_model, time, concentration, maxfev=10000)
    popt_p, _ = curve_fit(peppas_model, time, concentration, maxfev=10000)
    popt_f, _ = curve_fit(fick_model, time, concentration, maxfev=10000)
    return popt_h, popt_p, popt_f

def fit_and_evaluate(time, concentration, num_points):
    t_sub = time[:num_points]
    c_sub = concentration[:num_points]
    popt_h, _ = curve_fit(higuchi_model, t_sub, c_sub, maxfev=10000)
    popt_p, _ = curve_fit(peppas_model, t_sub, c_sub, maxfev=10000)
    popt_f, _ = curve_fit(fick_model, t_sub, c_sub,
                          p0=[0.1, np.max(c_sub)], maxfev=10000)
    return (higuchi_model(time, *popt_h),
            peppas_model(time, *popt_p),
            fick_model(time, *popt_f))

def calculate_errors(true_vals, predicted_vals):
    true_vals = np.asarray(true_vals).flatten()
    predicted_vals = np.asarray(predicted_vals).flatten()
    mae = np.mean(np.abs(true_vals - predicted_vals))
    rmse = np.sqrt(np.mean((true_vals - predicted_vals) ** 2))
    return mae, rmse

In [ ]:
classical_fits = {}
errors_classical = []

for label, conc in zip(film_labels, film_data_1d):
    popt_h, popt_p, popt_f = fit_models(time_normalized, conc)

    h_fit = higuchi_model(time_normalized, *popt_h)
    p_fit = peppas_model(time_normalized, *popt_p)
    f_fit = fick_model(time_normalized, *popt_f)
    classical_fits[label] = (h_fit, p_fit, f_fit)

    for model_name, fit_vals in [('Higuchi', h_fit), ('Peppas', p_fit), ('Fick', f_fit)]:
        mae, rmse = calculate_errors(conc, fit_vals)
        errors_classical.append([label, model_name, mae, rmse])

errors_df = pd.DataFrame(errors_classical, columns=['Film Type', 'Model', 'MAE', 'RMSE'])
print('Classical model errors:')
display(errors_df)

## PINN

Inputs are normalized (x, t) and the output is the concentration u. The loss is the mean squared error on the measurements plus the mean squared Fick residual $f = u_t - D \, u_{xx}$ at 10,000 Latin hypercube collocation points. Five hidden layers of 20 tanh units, Adam with learning rate 1e-3, 2,500 epochs.

In [ ]:
D_COEFF = 0.01
LAYERS = [2, 20, 20, 20, 20, 20, 1]

def hyper_initial(size):
    in_dim, out_dim = size
    std = np.sqrt(2.0 / (in_dim + out_dim))
    return tf.Variable(tf.random.truncated_normal(shape=size, stddev=std))

def DNN(X, W, b, keep_prob=None):
    A = X
    for i in range(len(W) - 1):
        A = tf.tanh(tf.add(tf.matmul(A, W[i]), b[i]))
        # keep_prob turns on Monte Carlo dropout after every hidden layer (the BPINN)
        if keep_prob is not None:
            A = tf.nn.dropout(A, rate=1 - keep_prob)
    return tf.add(tf.matmul(A, W[-1]), b[-1])

def net_u(x, t, W, b, keep_prob=None):
    return DNN(tf.concat([x, t], 1), W, b, keep_prob)

def net_f(x, t, W, b, D, keep_prob=None):
    """Fick's law residual f = u_t - D u_xx by nested automatic differentiation"""
    with tf.GradientTape(persistent=True) as tape1:
        tape1.watch([x, t])
        with tf.GradientTape(persistent=True) as tape2:
            tape2.watch([x, t])
            u = net_u(x, t, W, b, keep_prob)
        u_t = tape2.gradient(u, t)
        u_x = tape2.gradient(u, x)
    u_xx = tape1.gradient(u_x, x)
    return u_t - D * u_xx

def train_step(W, b, X_u_tf, u_tf, X_f_tf, opt, D, keep_prob=None):
    """One Adam step on the data loss plus the Fick residual loss"""
    x_u, t_u = X_u_tf[:, 0:1], X_u_tf[:, 1:2]
    x_f, t_f = X_f_tf[:, 0:1], X_f_tf[:, 1:2]
    with tf.GradientTape() as tape:
        tape.watch(W + b)
        u_pred = net_u(x_u, t_u, W, b, keep_prob)
        f_pred = net_f(x_f, t_f, W, b, D, keep_prob)
        loss = tf.reduce_mean(tf.square(u_pred - u_tf)) + tf.reduce_mean(tf.square(f_pred))
    grads = tape.gradient(loss, W + b)
    opt.apply_gradients(zip(grads, W + b))
    return loss

def create_model_vars(layers=LAYERS):
    W = [hyper_initial([layers[l-1], layers[l]]) for l in range(1, len(layers))]
    b = [tf.Variable(tf.zeros([1, layers[l]])) for l in range(1, len(layers))]
    return W, b

def pinn_predict(t_col, W, b):
    X = np.hstack((t_col, np.zeros_like(t_col)))
    X_tf = tf.convert_to_tensor(X, dtype=tf.float32)
    return net_u(X_tf[:, 0:1], X_tf[:, 1:2], W, b).numpy()

In [ ]:
N_f = 10000
X_f_train = lhs(2, N_f)
X_f_tf = tf.convert_to_tensor(X_f_train, dtype=tf.float32)

EPOCHS_CLASSICAL = 2500

pinn_weights = {}

for label, u_train in zip(film_labels, film_data):
    W, b = create_model_vars()
    optimizer = tf.optimizers.Adam(learning_rate=1e-3)
    X_u = np.hstack((t_col, np.zeros_like(t_col)))
    X_u_tf = tf.convert_to_tensor(X_u, dtype=tf.float32)
    u_tf = tf.convert_to_tensor(u_train, dtype=tf.float32)

    for epoch in range(EPOCHS_CLASSICAL):
        loss = train_step(W, b, X_u_tf, u_tf, X_f_tf, optimizer, D_COEFF)
        if epoch % 500 == 0:
            print(f'{label} | Epoch {epoch:>4d} | Loss: {loss.numpy():.6f}')

    pinn_weights[label] = (W, b)
    print(f'{label} film training complete.\n')

In [ ]:
pinn_preds = {}
for label, u_train in zip(film_labels, film_data):
    W, b = pinn_weights[label]
    pred = pinn_predict(t_col, W, b)
    pinn_preds[label] = pred

    mae, rmse = calculate_errors(u_train, pred)
    errors_df = pd.concat([errors_df, pd.DataFrame(
        [[label, 'PINN', mae, rmse]], columns=['Film Type', 'Model', 'MAE', 'RMSE']
    )], ignore_index=True)

print('All model errors (classical + PINN):')
display(errors_df)

## Figure 3: PINN and classical model release curves

In [ ]:
fig = plt.figure(figsize=(10, 8))

subplot_pos = [(2, 2, 1), (2, 2, 2), (2, 2, (3, 4))]
titles = ['Flat Film', '1D Wrinkled Film', '2D Crumpled Film']

for idx, (label, conc) in enumerate(zip(film_labels, film_data_1d)):
    ax = fig.add_subplot(*subplot_pos[idx])
    h_fit, p_fit, f_fit = classical_fits[label]
    pred = pinn_preds[label].flatten()

    ax.plot(time_normalized, conc, 'ko', markersize=5, label='Experimental Data')
    ax.plot(time_normalized, h_fit, '-', label='Higuchi Model')
    ax.plot(time_normalized, p_fit, '--', label='Peppas Model')
    ax.plot(time_normalized, f_fit, '-.', label="Fick's Law Model")
    ax.plot(time_normalized, pred, ':', linewidth=2, label='PINN')
    ax.set_title(titles[idx])
    ax.set_ylabel('Concentration (ppm)')
    if idx == 2:
        ax.set_xlabel('Normalized Time')
    ax.legend(loc='lower right', fontsize=9)

fig.tight_layout()
fig.savefig('figures/figure_3.png', bbox_inches='tight')
plt.show()
print('Saved: figures/figure_3.png')

## Figure 4: MAE and RMSE

In [ ]:
plot_df = errors_df.copy()
plot_df['Model'] = pd.Categorical(plot_df['Model'],
                                   categories=['Higuchi', 'Peppas', 'Fick', 'PINN'], ordered=True)
plot_df['Film Type'] = pd.Categorical(plot_df['Film Type'],
                                       categories=['Flat', '1D', '2D'], ordered=True)

fig, axes = plt.subplots(2, 1, figsize=(8, 6))

plot_df.pivot(index='Model', columns='Film Type', values='MAE').plot(kind='bar', ax=axes[0])
axes[0].set_title('Mean Absolute Error (MAE)')
axes[0].set_ylabel('MAE')
axes[0].legend(title='Film Type')
axes[0].tick_params(axis='x', rotation=0)

plot_df.pivot(index='Model', columns='Film Type', values='RMSE').plot(kind='bar', ax=axes[1])
axes[1].set_title('Root Mean Square Error (RMSE)')
axes[1].set_ylabel('RMSE')
axes[1].legend(title='Film Type')
axes[1].tick_params(axis='x', rotation=0)

fig.tight_layout()
fig.savefig('figures/figure_4.png', bbox_inches='tight')
plt.show()
print('Saved: figures/figure_4.png')

## Figure 5: predicted profiles u(x, t)

The trained networks evaluated over $x \in [0, 1]$ at normalized times $t = 0$, $0.25$ and $0.5$.

In [ ]:
x_dense = np.linspace(0, 1, 200)[:, np.newaxis]
time_stamps = [0.0, 0.25, 0.5]

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for idx, (label, title) in enumerate(zip(film_labels, titles)):
    ax = axes[idx]
    W, b = pinn_weights[label]

    for t_val in time_stamps:
        t_arr = np.full_like(x_dense, t_val)
        X_input = tf.convert_to_tensor(np.hstack((x_dense, t_arr)), dtype=tf.float32)
        u_pred = net_u(X_input[:, 0:1], X_input[:, 1:2], W, b).numpy().flatten()
        ax.plot(x_dense.flatten(), u_pred, label=f't = {t_val}')

    ax.set_title(title)
    ax.set_xlabel('Spatial coordinate x')
    if idx == 0:
        ax.set_ylabel('Predicted concentration u(x, t)')
    ax.legend()

fig.suptitle('Spatial concentration profiles at selected time stamps', y=1.02, fontsize=14)
fig.tight_layout()
fig.savefig('figures/figure_5.png', bbox_inches='tight')
plt.show()
print('Saved: figures/figure_5.png')

## Noise study: ensemble of 50 PINNs

Gaussian noise ($\mu = 0$, $\sigma = 0.05$) is added to the measurements, and 50 PINNs are trained for 5,000 epochs each with different seeds and noise draws. The ensemble mean and standard deviation give the PINN uncertainty band.

In [ ]:
def add_gaussian_noise(data, mean=0, std=0.05):
    return data + np.random.normal(mean, std, data.shape)

def train_single_pinn(t_col, u_noisy, epochs=5000, seed=None):
    if seed is not None:
        np.random.seed(seed)
        tf.random.set_seed(seed)
    W, b = create_model_vars()
    opt = tf.optimizers.Adam(learning_rate=1e-3)

    X_u = np.hstack((t_col, np.zeros_like(t_col)))
    X_u_tf = tf.convert_to_tensor(X_u, dtype=tf.float32)
    u_tf = tf.convert_to_tensor(u_noisy, dtype=tf.float32)

    for _ in range(epochs):
        train_step(W, b, X_u_tf, u_tf, X_f_tf, opt, D_COEFF)

    return pinn_predict(t_col, W, b).flatten()

N_ENSEMBLE = 50
EPOCHS_ENSEMBLE = 5000
ensemble_preds = {label: [] for label in film_labels}

for i in range(N_ENSEMBLE):
    if (i + 1) % 10 == 0 or i == 0:
        print(f'Training PINN {i+1}/{N_ENSEMBLE}')
    seed = random.randint(0, 10000)
    for label, u_train in zip(film_labels, film_data):
        u_noisy = add_gaussian_noise(u_train)
        pred = train_single_pinn(t_col, u_noisy, epochs=EPOCHS_ENSEMBLE, seed=seed)
        ensemble_preds[label].append(pred)

ensemble_mean, ensemble_std = {}, {}
for label in film_labels:
    arr = np.array(ensemble_preds[label])
    ensemble_mean[label] = np.mean(arr, axis=0)
    ensemble_std[label] = np.std(arr, axis=0)

print('\nEnsemble PINN errors (mean of 50 runs):')
pinn_noise_errors = []
for label, u_train in zip(film_labels, film_data):
    mae, rmse = calculate_errors(u_train.flatten(), ensemble_mean[label])
    pinn_noise_errors.append([label, 'PINN', mae, rmse])
    print(f'  {label}: MAE={mae:.4f}, RMSE={rmse:.4f}')

noise_errors_df = pd.DataFrame(pinn_noise_errors, columns=['Film Type', 'Model', 'MAE', 'RMSE'])

## Noise study: BPINN with Monte Carlo dropout

The same network with dropout (keep probability 0.975) after every hidden layer, trained for 10,000 epochs on noisy data. 100 stochastic forward passes give the predictive mean and standard deviation.

In [ ]:
KEEP_PROB = 0.975

def train_bpinn(t_col, u_noisy, epochs=10000):
    W, b = create_model_vars()
    opt = tf.optimizers.Adam(learning_rate=1e-3)
    X_u = np.hstack((t_col, np.zeros_like(t_col)))
    X_u_tf = tf.convert_to_tensor(X_u, dtype=tf.float32)
    u_tf = tf.convert_to_tensor(u_noisy, dtype=tf.float32)

    for epoch in range(epochs):
        loss = train_step(W, b, X_u_tf, u_tf, X_f_tf, opt, D_COEFF, KEEP_PROB)
        if epoch % 2000 == 0:
            print(f'  BPINN epoch {epoch:>5d} | loss: {loss.numpy():.6f}')
    return W, b

def predict_bpinn(W, b, t_col, n_samples=100):
    """Mean and standard deviation over n_samples stochastic forward passes"""
    X = np.hstack((t_col, np.zeros_like(t_col)))
    X_tf = tf.convert_to_tensor(X, dtype=tf.float32)
    preds = [net_u(X_tf[:, 0:1], X_tf[:, 1:2], W, b, KEEP_PROB) for _ in range(n_samples)]
    preds = tf.stack(preds, axis=0)
    return tf.reduce_mean(preds, axis=0).numpy().flatten(), tf.math.reduce_std(preds, axis=0).numpy().flatten()

In [ ]:
bpinn_results = {}
bpinn_noise_errors = []

for label, u_train in zip(film_labels, film_data):
    print(f'Training BPINN for {label} film...')
    u_noisy = add_gaussian_noise(u_train)
    W_b, b_b = train_bpinn(t_col, u_noisy, epochs=10000)
    mean_pred, std_pred = predict_bpinn(W_b, b_b, t_col)
    bpinn_results[label] = (mean_pred, std_pred)

    mae, rmse = calculate_errors(u_train.flatten(), mean_pred)
    bpinn_noise_errors.append([label, 'BPINN', mae, rmse])
    print(f'  {label} BPINN: MAE={mae:.4f}, RMSE={rmse:.4f}\n')

bpinn_errors_df = pd.DataFrame(bpinn_noise_errors, columns=['Film Type', 'Model', 'MAE', 'RMSE'])
print('BPINN errors:')
display(bpinn_errors_df)

## Figure 6: PINN ensemble and BPINN uncertainty under noise

In [ ]:
fig = plt.figure(figsize=(10, 8))
t_flat = t_col.flatten()

for idx, (label, u_true) in enumerate(zip(film_labels, film_data)):
    ax = fig.add_subplot(*subplot_pos[idx])
    u_true_flat = u_true.flatten()

    ax.plot(t_flat, u_true_flat, 'ko', markersize=5, label='Experimental Data')

    m_e, s_e = ensemble_mean[label], ensemble_std[label]
    ax.plot(t_flat, m_e, '-', color='red', label='PINN Ensemble Mean')
    ax.fill_between(t_flat, m_e - s_e, m_e + s_e, color='red', alpha=0.2,
                    label='PINN $\\pm$ 1 std')

    m_b, s_b = bpinn_results[label]
    ax.plot(t_flat, m_b, '-', color='green', label='BPINN Mean')
    ax.fill_between(t_flat, m_b - s_b, m_b + s_b, color='green', alpha=0.2,
                    label='BPINN $\\pm$ 1 std')

    ax.set_title(titles[idx])
    ax.set_ylabel('Concentration (ppm)')
    if idx == 2:
        ax.set_xlabel('Normalized Time')
        ax.legend(loc='lower right', fontsize=8)

fig.tight_layout()
fig.savefig('figures/figure_6.png', bbox_inches='tight')
plt.show()
print('Saved: figures/figure_6.png')

## Figure 7: MAE and RMSE under noise

In [ ]:
combined_noise = pd.concat([noise_errors_df, bpinn_errors_df], ignore_index=True)
combined_noise['Model'] = pd.Categorical(combined_noise['Model'],
                                          categories=['PINN', 'BPINN'], ordered=True)
combined_noise['Film Type'] = pd.Categorical(combined_noise['Film Type'],
                                              categories=['Flat', '1D', '2D'], ordered=True)

fig, axes = plt.subplots(2, 1, figsize=(8, 6))

combined_noise.pivot(index='Model', columns='Film Type', values='MAE').plot(kind='bar', ax=axes[0])
axes[0].set_title('MAE Under Noisy Conditions')
axes[0].set_ylabel('MAE')
axes[0].legend(title='Film Type')
axes[0].tick_params(axis='x', rotation=0)

combined_noise.pivot(index='Model', columns='Film Type', values='RMSE').plot(kind='bar', ax=axes[1])
axes[1].set_title('RMSE Under Noisy Conditions')
axes[1].set_ylabel('RMSE')
axes[1].legend(title='Film Type')
axes[1].tick_params(axis='x', rotation=0)

fig.tight_layout()
fig.savefig('figures/figure_7.png', bbox_inches='tight')
plt.show()
print('Saved: figures/figure_7.png')

## Limited data

PINNs (2,000 epochs, 1,000 collocation points) and the classical models are trained on the first n measurements, n = 2 to 14, and scored on all 14.

In [ ]:
def train_pinn_limited(t_col, u_train, n, epochs=2000):
    W, b = create_model_vars()
    opt = tf.optimizers.Adam(learning_rate=1e-3)

    t_sub = t_col[:n]
    u_sub = u_train[:n]
    N_f_limited = 1000
    X_f = tf.convert_to_tensor(lhs(2, N_f_limited), dtype=tf.float32)

    X_u = np.hstack((t_sub, np.zeros_like(t_sub)))
    X_u_tf = tf.convert_to_tensor(X_u, dtype=tf.float32)
    u_tf = tf.convert_to_tensor(u_sub, dtype=tf.float32)

    for epoch in range(epochs):
        train_step(W, b, X_u_tf, u_tf, X_f, opt, D_COEFF)

    return pinn_predict(t_col, W, b)

limited_pinn_preds = {label: [] for label in film_labels}

for n in range(2, 15):
    print(f'Training limited-data PINNs with n={n}...')
    for label, u_train in zip(film_labels, film_data):
        pred = train_pinn_limited(t_col, u_train, n)
        limited_pinn_preds[label].append(pred)

print('Limited-data training complete.')

In [ ]:
limited_errors = []

for n in range(2, 15):
    for label, conc_1d in zip(film_labels, film_data_1d):
        h_fit, p_fit, f_fit = fit_and_evaluate(time_normalized, conc_1d, n)

        for model_name, fit_vals in [('Higuchi', h_fit), ('Peppas', p_fit), ('Fick', f_fit)]:
            rmse_val = np.sqrt(mean_squared_error(conc_1d, fit_vals))
            mae_val = mean_absolute_error(conc_1d, fit_vals)
            limited_errors.append({'n': n, 'Film': label, 'Model': model_name,
                                   'RMSE': rmse_val, 'MAE': mae_val})

        pinn_pred = limited_pinn_preds[label][n - 2].flatten()
        rmse_val = np.sqrt(mean_squared_error(conc_1d, pinn_pred))
        mae_val = mean_absolute_error(conc_1d, pinn_pred)
        limited_errors.append({'n': n, 'Film': label, 'Model': 'PINN',
                               'RMSE': rmse_val, 'MAE': mae_val})

limited_errors_df = pd.DataFrame(limited_errors)
print('Sample of limited-data errors:')
display(limited_errors_df.head(12))

## Figure 8: predictions at the smallest n with PINN RMSE < 0.05

Flat film n = 9 (first 120 min), 1D and 2D films n = 11 (first 720 min).

In [ ]:
best_n = {'Flat': 9, '1D': 11, '2D': 11}
fig = plt.figure(figsize=(10, 8))

for idx, (label, conc) in enumerate(zip(film_labels, film_data_1d)):
    ax = fig.add_subplot(*subplot_pos[idx])
    n = best_n[label]

    h_fit, p_fit, f_fit = fit_and_evaluate(time_normalized, conc, n)
    pinn_pred = limited_pinn_preds[label][n - 2].flatten()

    ax.plot(time_normalized, conc, 'ko', markersize=5, label='Experimental Data')
    ax.plot(time_normalized, h_fit, '-', label='Higuchi Model')
    ax.plot(time_normalized, p_fit, '--', label='Peppas Model')
    ax.plot(time_normalized, f_fit, '-.', label="Fick's Law Model")
    ax.plot(time_normalized, pinn_pred, ':', linewidth=2, label=f'PINN (n={n})')

    ax.axvline(x=time_normalized[n-1], color='gray', linestyle='--', alpha=0.5,
               label=f'Training cutoff (n={n})')

    ax.set_title(f'{titles[idx]} (n={n})')
    ax.set_ylabel('Concentration (ppm)')
    if idx == 2:
        ax.set_xlabel('Normalized Time')
    ax.legend(loc='lower right', fontsize=8)

fig.tight_layout()
fig.savefig('figures/figure_8.png', bbox_inches='tight')
plt.show()
print('Saved: figures/figure_8.png')

## Figure 9: RMSE and MAE against n

The dashed line marks RMSE = 0.05.

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(12, 12))
y_max = 0.30
models_list = ['Higuchi', 'Peppas', 'Fick', 'PINN']
styles = [('-', 'o'), ('--', 'x'), ('-.', 's'), (':', 'd')]

for row, label in enumerate(film_labels):
    film_df = limited_errors_df[limited_errors_df['Film'] == label]

    for col, metric in enumerate(['RMSE', 'MAE']):
        ax = axs[row, col]
        for model_name, (ls, mk) in zip(models_list, styles):
            subset = film_df[film_df['Model'] == model_name]
            ax.plot(subset['n'], subset[metric], linestyle=ls, marker=mk,
                    markersize=5, label=model_name)

        if col == 0:
            ax.axhline(y=0.05, color='gray', linestyle='--', alpha=0.6, label='RMSE = 0.05')

        ax.set_ylim(0, y_max)
        ax.set_title(f'{titles[row]} {metric}')
        ax.set_xlabel('n (Data Points)')
        ax.set_ylabel(metric)
        ax.legend(fontsize=8)

fig.tight_layout()
fig.savefig('figures/figure_9.png', bbox_inches='tight')
plt.show()
print('Saved: figures/figure_9.png')